# Population mean firing rate: stimulation vs control trials

A control analysis for the coding-direction projection work. Before reading a
CD projection as "stimulation pushes activity toward the no-response / no-reward
pole", we need to rule out the simplest confound: **stimulation just changes the
overall firing rate of the selected population**. A global suppression (or boost)
would move the population along essentially any axis.

Here we compute, for the **same selected population** used in the CD notebook,
the mean firing rate across units (one value per trial, time-resolved) and
compare **stimulation** trials against **control** (non-laser) trials:

- **Time-resolved** — population mean ± SEM trace for stim vs control.
- **Window-averaged** — per-trial scalar (mean rate in a response window),
  compared with a Mann–Whitney U test and shown as a strip plot.
- **Per stimulation condition** — the same comparison for each laser condition.

Set `ZSCORE_UNITS = True` to give every unit equal weight (otherwise the raw-Hz
mean is dominated by the highest-firing units).

In [ ]:
# =============================================================================
# 1. ENVIRONMENT SETUP
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"✅ Analysis modules loaded from: {MODULE_PATH}")

In [ ]:
# =============================================================================
# 2. CONFIG
# =============================================================================
import re
from pathlib import Path

# --- Session identifiers -----------------------------------------------------
session_date = '863973_2026-08-25_14-18-29'
ephys_session = None   # None -> auto-discover from the PSTH Zarr in step 3

# --- Paths -------------------------------------------------------------------
PSTH_DIR = Path("/root/capsule/scratch/psth_results")
RESULTS_DIR = Path("/root/capsule/scratch/behavior_summary")

# --- Unit selection (same combinable filters as the CD notebook) -------------
USE_TAGGED = False     # True -> restrict to opto-tagged units
PROBES = ["ProbeA-1", "ProbeA-2", "ProbeA-3", "ProbeA-4"]   # None/[] -> all probes

# Opto-tagging source + condition (only used when USE_TAGGED is True).
metrics_csv_path = Path(f"/root/capsule/scratch/opto_tagging/metrics_{session_date}.csv")
#   (target_power, location_tag, laser_name, cycle_duration, frequency, pulse_duration)
target_cond = (5.0, 1.0, "Laser_1", 1.0, 10.0, 0.05)
SELECT_PULSE_INDEX = -1

# --- Population-mean parameters ----------------------------------------------
ALIGN_TO = "go_cue"        # alignment event
TRACE_WINDOW = (-4.0, 6.0) # time range (s) for the time-resolved trace
STAT_WINDOW = (0.0, 0.5)   # window (s) for the per-trial scalar comparison
ZSCORE_UNITS = False       # True -> z-score each unit before averaging (equal weight)

_tag_part = "tagged" if USE_TAGGED else "allqc"
_probe_part = "all_probes" if not PROBES else "probe_" + "_".join(
    re.sub(r"[^0-9A-Za-z._-]", "", str(p)) for p in PROBES
)
OUTDIR = Path(f"/root/capsule/scratch/population_mean/{session_date}/{_tag_part}__{_probe_part}")

print(f"Use tagged : {USE_TAGGED}")
print(f"Probes     : {'all' if not PROBES else PROBES}")
print(f"Z-score    : {ZSCORE_UNITS}")
print(f"Output dir : {OUTDIR}")

In [ ]:
# =============================================================================
# 3. LOAD SESSION RESOURCES (Zarr PSTH, NWB, trials) + define stim / control
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt
from behavior_utils import find_trials
from nwb_utils import NWBUtils
from create_psth import load_zarr
from ephys_while_stimulation import summarize_laser_conditions

if ephys_session is None:
    cands = sorted(PSTH_DIR.glob(f"ecephys_{session_date}_sorted*_0.2s.zarr"))
    if not cands:
        raise FileNotFoundError(
            f"No PSTH Zarr matching ecephys_{session_date}_sorted*_0.2s.zarr in {PSTH_DIR}"
        )
    ephys_session = cands[0].name[: -len("_0.2s.zarr")]
print(f"ephys_session: {ephys_session}")

zarr_path = PSTH_DIR / f"{ephys_session}_0.2s.zarr"
assert zarr_path.exists(), f"Zarr not found: {zarr_path}"

nwb, _ = NWBUtils.combine_nwb(session_name=ephys_session)
psth = load_zarr(str(zarr_path))

# Laser-on trials -> stimulation set; the rest are the control (non-laser) set.
n_trials = int(np.asarray(nwb.trials["animal_response"][:]).size)
all_trials = np.arange(n_trials, dtype=np.int64)
if "laser_on_trial" in nwb.trials.colnames:
    laser_on = np.asarray(nwb.trials["laser_on_trial"][:])
    stim_trials = np.where(laser_on == 1)[0].astype(np.int64)
else:
    stim_trials = np.array([], dtype=np.int64)
control_trials = np.setdiff1d(all_trials, stim_trials)
print(f"stim (laser-on) trials : {stim_trials.size}")
print(f"control (non-laser)    : {control_trials.size}")

# Enumerate stimulation conditions (same grouping as the CD notebook).
STIM_GROUP_COLS = (
    "laser_start", "laser_start_offset", "laser_end", "laser_end_offset", "laser_duration",
)
stim_summary = summarize_laser_conditions(nwb, group_cols=STIM_GROUP_COLS)
stim_conditions = {
    i: np.asarray(row["trial_ids"], dtype=np.int64)
    for i, row in stim_summary.iterrows()
}
print(f"{len(stim_conditions)} stimulation condition(s)")

OUTDIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# =============================================================================
# 4. SELECT UNITS (same filters as the CD notebook)
# =============================================================================
from stim_axis_projection import select_units

unit_ids = select_units(
    nwb, psth, use_tagged=USE_TAGGED, probes=PROBES,
    metrics_csv_path=metrics_csv_path, target_cond=target_cond,
    pulse_index=SELECT_PULSE_INDEX,
)
unit_ids = np.asarray(unit_ids, dtype=np.int64)
print(f"Selected units ({unit_ids.size}): {unit_ids.tolist()}")
if unit_ids.size < 1:
    raise ValueError("No units after selection — loosen the filters.")

## 5. Population mean trace: stim vs control

Extract the trials × units × time tensor for the selected units, average over
units to get one population-mean time course per trial, then compare the mean ±
SEM across trials for the stim and control sets. If the stim trace sits below
(or above) the control trace around the stimulation window, that is a global
firing-rate effect that any CD projection would partly reflect.

In [ ]:
# =============================================================================
# 5. POPULATION MEAN PER TRIAL (selected units) + stim vs control trace
# =============================================================================
from ephys_dimension_reduction_CD import extract_trial_unit_timecube

# trials x units x time for the selected units, over the trace window.
out = extract_trial_unit_timecube(
    psth, align=ALIGN_TO, time_window=TRACE_WINDOW, unit_ids=unit_ids,
)
cube = out["cube"].astype(float)          # (T, N, Tt)
tids_all = np.asarray(out["trial_ids"], dtype=np.int64)
time = np.asarray(out["time"], dtype=float)
print(f"cube: {cube.shape} (trials x units x time)")

# Optional per-unit z-score (over all trials & time) so each unit weighs equally.
if ZSCORE_UNITS:
    mu = np.nanmean(cube, axis=(0, 2), keepdims=True)
    sd = np.nanstd(cube, axis=(0, 2), keepdims=True)
    sd[sd == 0] = 1.0
    cube = (cube - mu) / sd

pop = np.nanmean(cube, axis=1)            # (T, Tt) population mean per trial
_ylab = "population mean (z)" if ZSCORE_UNITS else "population mean rate (Hz)"

def _rows(ids):
    return np.isin(tids_all, np.asarray(ids, dtype=np.int64))

def _mean_sem(ids):
    m = pop[_rows(ids)]
    if m.shape[0] == 0:
        return None, None, 0
    return np.nanmean(m, axis=0), np.nanstd(m, axis=0) / np.sqrt(m.shape[0]), m.shape[0]

fig, ax = plt.subplots(figsize=(7, 4))
for ids, nm, c in [
    (control_trials, "control (non-laser)", "tab:gray"),
    (stim_trials, "stimulation", "tab:green"),
]:
    m, sem, n = _mean_sem(ids)
    if m is None:
        continue
    ax.plot(time, m, color=c, label=f"{nm} (n={n})")
    ax.fill_between(time, m - sem, m + sem, color=c, alpha=0.2)
ax.axvline(0, color="k", ls="--", lw=0.8)
ax.set_xlabel(f"Time from {ALIGN_TO} (s)")
ax.set_ylabel(_ylab)
ax.set_title("Population mean — stim vs control")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUTDIR / "popmean_trace_stim_vs_control.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =============================================================================
# 6. WINDOW-AVERAGED SCALAR COMPARISON (stim vs control) + Mann-Whitney
# =============================================================================
from scipy.stats import mannwhitneyu

wmask = (time >= STAT_WINDOW[0]) & (time < STAT_WINDOW[1])
if not np.any(wmask):
    raise ValueError(f"STAT_WINDOW {STAT_WINDOW} selects no time bins.")
scalar = np.nanmean(pop[:, wmask], axis=1)   # (T,) per-trial mean rate in window

s_stim = scalar[_rows(stim_trials)]
s_ctrl = scalar[_rows(control_trials)]
u, p = mannwhitneyu(s_stim, s_ctrl, alternative="two-sided")
print(f"window {STAT_WINDOW} s  ({_ylab})")
print(f"stim    : mean={np.nanmean(s_stim):.3f}  n={s_stim.size}")
print(f"control : mean={np.nanmean(s_ctrl):.3f}  n={s_ctrl.size}")
print(f"Mann-Whitney U={u:.0f}, p={p:.3g}")

fig, ax = plt.subplots(figsize=(5, 4))
rng = np.random.RandomState(0)
for i, (vals, nm, c) in enumerate([
    (s_ctrl, "control", "tab:gray"),
    (s_stim, "stim", "tab:green"),
]):
    if vals.size == 0:
        continue
    ax.scatter(np.full(vals.size, i) + rng.uniform(-0.09, 0.09, vals.size),
               vals, s=14, alpha=0.4, color=c)
    ax.hlines(np.nanmean(vals), i - 0.22, i + 0.22, color=c, lw=2.5)
ax.set_xticks([0, 1]); ax.set_xticklabels(["control", "stim"])
ax.set_ylabel(f"{_ylab}\n(mean over {STAT_WINDOW} s)")
ax.set_title(f"Window mean — Mann-Whitney p={p:.3g}")
fig.tight_layout()
fig.savefig(OUTDIR / "popmean_window_stim_vs_control.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Per stimulation condition

The same population-mean trace, but with each laser condition drawn separately
against the shared control set. This shows whether different stimulation
parameters produce different magnitudes of global rate change.

In [ ]:
# =============================================================================
# 7. POPULATION MEAN TRACE PER STIM CONDITION (vs control)
# =============================================================================
fig, ax = plt.subplots(figsize=(7, 4))

m, sem, n = _mean_sem(control_trials)
if m is not None:
    ax.plot(time, m, color="tab:gray", lw=1.6, label=f"control (n={n})")
    ax.fill_between(time, m - sem, m + sem, color="tab:gray", alpha=0.18)

_cmap = plt.get_cmap("viridis")
_cond_ids = sorted(stim_conditions)
for k, ci in enumerate(_cond_ids):
    col = _cmap(k / max(len(_cond_ids) - 1, 1))
    m, sem, n = _mean_sem(stim_conditions[ci])
    if m is None:
        continue
    ax.plot(time, m, color=col, label=f"stim cond {ci} (n={n})")
    ax.fill_between(time, m - sem, m + sem, color=col, alpha=0.15)

ax.axvline(0, color="k", ls="--", lw=0.8)
ax.set_xlabel(f"Time from {ALIGN_TO} (s)")
ax.set_ylabel(_ylab)
ax.set_title("Population mean — per stim condition vs control")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUTDIR / "popmean_trace_by_condition.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Sweep across unit selections

Repeat the stim-vs-control population-mean comparison for each unit selection —
**tagged-all**, **ProbeA**, **ProbeB**, **ProbeC** — in one figure (a panel per
selection). The companion table reports, per selection, the window-averaged mean
rate for stim and control and the Mann–Whitney p-value, so a global rate change
in a specific probe group is easy to spot.


In [ ]:
# =============================================================================
# 8. SWEEP: population mean (stim vs control) per unit selection
# =============================================================================
import pandas as pd
from scipy.stats import mannwhitneyu

UNIT_SELECTIONS = [
    ("tagged-all", dict(use_tagged=True,  probes=None)),
    ("ProbeA",     dict(use_tagged=False, probes=["ProbeA-1", "ProbeA-2", "ProbeA-3", "ProbeA-4"])),
    ("ProbeB",     dict(use_tagged=False, probes=["ProbeB-1", "ProbeB-2", "ProbeB-3", "ProbeB-4"])),
    ("ProbeC",     dict(use_tagged=False, probes=["ProbeC-1", "ProbeC-2", "ProbeC-3", "ProbeC-4"])),
]


def population_mean_per_trial(sel_unit_ids, window):
    """trials x units x time -> (trial_ids, time, population mean per trial)."""
    out = extract_trial_unit_timecube(
        psth, align=ALIGN_TO, time_window=window,
        unit_ids=np.asarray(sel_unit_ids, dtype=np.int64),
    )
    c = out["cube"].astype(float)
    if ZSCORE_UNITS:
        mu = np.nanmean(c, axis=(0, 2), keepdims=True)
        sd = np.nanstd(c, axis=(0, 2), keepdims=True)
        sd[sd == 0] = 1.0
        c = (c - mu) / sd
    return (np.asarray(out["trial_ids"], dtype=np.int64),
            np.asarray(out["time"], dtype=float),
            np.nanmean(c, axis=1))


# Resolve units once per selection.
units_by_sel = {}
for name, kw in UNIT_SELECTIONS:
    try:
        uids = np.asarray(select_units(
            nwb, psth, metrics_csv_path=metrics_csv_path,
            target_cond=target_cond, pulse_index=SELECT_PULSE_INDEX, **kw,
        ), dtype=np.int64)
    except Exception as e:
        print(f"[{name}] unit selection failed: {e}")
        uids = np.array([], dtype=np.int64)
    units_by_sel[name] = uids
    print(f"{name:12s}: {uids.size} units")

# --- grid of time-resolved traces -------------------------------------------
sel_names = [n for n, _ in UNIT_SELECTIONS]
ncol = len(sel_names)
fig, axs = plt.subplots(1, ncol, figsize=(3.6 * ncol, 3.2), sharex=True, squeeze=False)
rows = []
for j, name in enumerate(sel_names):
    ax = axs[0][j]
    uids = units_by_sel[name]
    if uids.size < 1:
        ax.text(0.5, 0.5, "no units", ha="center", va="center",
                transform=ax.transAxes, color="0.6")
        ax.set_xticks([]); ax.set_yticks([])
        continue
    tids_s, time_s, pop_s = population_mean_per_trial(uids, TRACE_WINDOW)

    def ms(ids):
        m = pop_s[np.isin(tids_s, np.asarray(ids, dtype=np.int64))]
        if m.shape[0] == 0:
            return None, None, 0
        return np.nanmean(m, axis=0), np.nanstd(m, axis=0) / np.sqrt(m.shape[0]), m.shape[0]

    for ids, nm, c in [
        (control_trials, "control", "tab:gray"),
        (stim_trials, "stim", "tab:green"),
    ]:
        m, sem, n = ms(ids)
        if m is None:
            continue
        ax.plot(time_s, m, color=c, lw=1.3, label=f"{nm} (n={n})")
        ax.fill_between(time_s, m - sem, m + sem, color=c, alpha=0.18)
    ax.axvline(0, color="k", ls="--", lw=0.7)
    ax.set_title(f"{name} ({uids.size}u)", fontsize=9)
    ax.set_xlabel(f"Time from {ALIGN_TO} (s)", fontsize=8)
    if j == 0:
        ax.set_ylabel(_ylab, fontsize=8)
    if j == ncol - 1:
        ax.legend(frameon=False, fontsize=7)

    # window-averaged scalar stats for the table
    wmask_s = (time_s >= STAT_WINDOW[0]) & (time_s < STAT_WINDOW[1])
    scal = np.nanmean(pop_s[:, wmask_s], axis=1)
    ss = scal[np.isin(tids_s, stim_trials)]
    sc = scal[np.isin(tids_s, control_trials)]
    if ss.size and sc.size:
        u, p = mannwhitneyu(ss, sc, alternative="two-sided")
    else:
        u, p = np.nan, np.nan
    rows.append(dict(
        selection=name, n_units=int(uids.size),
        stim_mean=round(float(np.nanmean(ss)), 3) if ss.size else np.nan,
        control_mean=round(float(np.nanmean(sc)), 3) if sc.size else np.nan,
        diff=round(float(np.nanmean(ss) - np.nanmean(sc)), 3) if ss.size and sc.size else np.nan,
        mannwhitney_p=p,
    ))

fig.suptitle("Population mean \u2014 stim vs control, by unit selection", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(OUTDIR / "popmean_trace_by_selection.png", dpi=300, bbox_inches="tight")
plt.show()

sweep_tbl = pd.DataFrame(rows)
sweep_tbl.to_csv(OUTDIR / "popmean_window_by_selection.csv", index=False)
print(f"window {STAT_WINDOW} s ({_ylab})")
display(sweep_tbl)


## 9. Sweep across unit selections × stimulation conditions

Full grid: **rows = unit selection** (tagged-all, ProbeA, ProbeB, ProbeC),
**columns = stimulation condition**. Each panel overlays that condition's
population-mean trace on the shared control trace, so you can see whether a
given probe group shows a global rate change for a specific laser condition.
The companion table reports the window-averaged stim/control means and the
Mann–Whitney p-value for every (selection, condition) cell.


In [ ]:
# =============================================================================
# 9. SWEEP: population mean per (unit selection x stim condition) vs control
# =============================================================================
_cond_ids = sorted(stim_conditions)
nrow, ncol = len(sel_names), len(_cond_ids)
_cmap = plt.get_cmap("viridis")

fig, axs = plt.subplots(
    nrow, ncol, figsize=(3.4 * ncol, 2.7 * nrow),
    sharex=True, sharey=True, squeeze=False,
)
cond_rows = []
for r, name in enumerate(sel_names):
    uids = units_by_sel[name]
    if uids.size >= 1:
        tids_s, time_s, pop_s = population_mean_per_trial(uids, TRACE_WINDOW)
        wmask_s = (time_s >= STAT_WINDOW[0]) & (time_s < STAT_WINDOW[1])
        scal = np.nanmean(pop_s[:, wmask_s], axis=1)
    else:
        tids_s = time_s = pop_s = scal = None

    def ms(ids):
        m = pop_s[np.isin(tids_s, np.asarray(ids, dtype=np.int64))]
        if m.shape[0] == 0:
            return None, None, 0
        return np.nanmean(m, axis=0), np.nanstd(m, axis=0) / np.sqrt(m.shape[0]), m.shape[0]

    for c, ci in enumerate(_cond_ids):
        ax = axs[r][c]
        if pop_s is None:
            ax.text(0.5, 0.5, "no units", ha="center", va="center",
                    transform=ax.transAxes, color="0.6")
            ax.set_xticks([]); ax.set_yticks([])
            continue
        col = _cmap(c / max(ncol - 1, 1))
        mc, semc, nc = ms(control_trials)
        if mc is not None:
            ax.plot(time_s, mc, color="tab:gray", lw=1.2, label=f"control (n={nc})")
            ax.fill_between(time_s, mc - semc, mc + semc, color="tab:gray", alpha=0.15)
        ms_m, ms_sem, ms_n = ms(stim_conditions[ci])
        if ms_m is not None:
            ax.plot(time_s, ms_m, color=col, lw=1.3, label=f"stim (n={ms_n})")
            ax.fill_between(time_s, ms_m - ms_sem, ms_m + ms_sem, color=col, alpha=0.18)
        ax.axvline(0, color="k", ls="--", lw=0.7)
        if r == 0:
            ax.set_title(f"cond {ci}", fontsize=9)
        if c == 0:
            ax.set_ylabel(f"{name}\n{_ylab}", fontsize=8)
        if r == nrow - 1:
            ax.set_xlabel(f"Time from {ALIGN_TO} (s)", fontsize=8)
        if r == 0 and c == ncol - 1:
            ax.legend(frameon=False, fontsize=7)

        # window stats for this (selection, condition) cell
        ss = scal[np.isin(tids_s, stim_conditions[ci])]
        sc = scal[np.isin(tids_s, control_trials)]
        if ss.size and sc.size:
            u, p = mannwhitneyu(ss, sc, alternative="two-sided")
        else:
            u, p = np.nan, np.nan
        cond_rows.append(dict(
            selection=name, condition=ci, n_units=int(uids.size), n_stim=int(ss.size),
            stim_mean=round(float(np.nanmean(ss)), 3) if ss.size else np.nan,
            control_mean=round(float(np.nanmean(sc)), 3) if sc.size else np.nan,
            diff=round(float(np.nanmean(ss) - np.nanmean(sc)), 3) if ss.size and sc.size else np.nan,
            mannwhitney_p=p,
        ))

fig.suptitle("Population mean \u2014 stim condition vs control, by unit selection", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.savefig(OUTDIR / "popmean_trace_by_selection_condition.png", dpi=300, bbox_inches="tight")
plt.show()

cond_tbl = pd.DataFrame(cond_rows)
cond_tbl.to_csv(OUTDIR / "popmean_window_by_selection_condition.csv", index=False)
print(f"window {STAT_WINDOW} s ({_ylab})")
display(cond_tbl)
